# Reinforcement Learning Track · Policy Gradient Methods

**Track:** Reinforcement Learning  
**Objective:** Master the principles and applications of Policy Gradient Methods.

---



### 🎓 Junior to Senior: Concept Bridge

**The Senior Perspective:** Policy gradient methods are the backbone of modern RL. PPO (a PG method) trains the RL step of RLHF. Understanding the policy gradient theorem explains exactly HOW a reward signal teaches an LLM to be helpful — the math is identical.

**Mental Model:** Imagine learning to play darts. Value-based is like calculating the exact score of every possible throw angle, then choosing the best one. Policy-based is like throwing darts, seeing where they land, and gradually adjusting your throwing motion. You don't need to evaluate every angle — you directly improve your technique from experience.

**Common Junior Pitfall:** Not using a baseline. Raw REINFORCE has enormous variance — the learning curve bounces wildly. Adding a baseline (subtracting the average return) dramatically reduces variance without introducing bias.

---

## 📑 Table of Contents

  * [🎓 Junior to Senior: Concept Bridge](#junior-to-senior-concept-bridge)
* [1 · The Policy Gradient Theorem](#1-the-policy-gradient-theorem)
  * [The Core Idea](#the-core-idea)
  * [The Objective](#the-objective)
  * [The Gradient (How to Improve)](#the-gradient-how-to-improve)
* [2 · REINFORCE Algorithm](#2-reinforce-algorithm)
* [3 · Actor-Critic: Best of Both Worlds](#3-actor-critic-best-of-both-worlds)
  * [The Advantage Function](#the-advantage-function)
  * [Preventing Deterministic Collapse (Entropy Bonus)](#preventing-deterministic-collapse-entropy-bonus)
* [🔨 Practical Practice](#practical-practice)
  * [Exercise 1: Real-Time TD-Advantage](#exercise-1-real-time-td-advantage)
  * [Exercise 2: Tune the Entropy Bonus](#exercise-2-tune-the-entropy-bonus)
  * [Exercise 3: Continuous BipedalWalker](#exercise-3-continuous-bipedalwalker)


In [ ]:
!pip install -q torch numpy matplotlib gymnasium

## 1 · The Policy Gradient Theorem

### The Core Idea

Instead of learning values, parameterize the POLICY directly:

$$\pi_\theta(a|s) = \text{probability of taking action } a \text{ in state } s$$

For a neural network: input is state $s$, output is a probability distribution over actions (via softmax).

### The Objective

Maximize the expected total reward:

$$J(\theta) = \mathbb{E}_{\pi_\theta} \left[ \sum_{t=0}^T r_t \right]$$

### The Gradient (How to Improve)

$$\boxed{\nabla_\theta J(\theta) = \mathbb{E} \left[ \sum_{t=0}^T \nabla_\theta \log \pi_\theta(a_t|s_t) \cdot G_t \right]}$$

In plain English:
- $\log \pi_\theta(a_t|s_t)$: the log-probability of the action we took
- $G_t$: the return (how good was the outcome?)
- If $G_t$ is HIGH: increase the probability of that action (it worked!)
- If $G_t$ is LOW: decrease the probability (it failed!)

**This is EXACTLY how RLHF works**: the "return" is the reward model's score. If the LLM's response gets a high score → increase the probability of that response. Low score → decrease it.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt
import gymnasium as gym

class DiscretePolicyNetwork(nn.Module):
    """Policy for DISCRETE action spaces (e.g., left/right).
    Outputs categorical probabilities via Softmax.
    """
    def __init__(self, state_dim, action_dim, hidden=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(state_dim, hidden),
            nn.ReLU(),
            nn.Linear(hidden, hidden),
            nn.ReLU(),
            nn.Linear(hidden, action_dim)
        )
    
    def forward(self, state):
        logits = self.net(state)
        return torch.softmax(logits, dim=-1)
    
    def act(self, state):
        state_t = torch.FloatTensor(state).unsqueeze(0)
        probs = self.forward(state_t)
        dist = torch.distributions.Categorical(probs)
        action = dist.sample()
        return action.item(), dist.log_prob(action)

class ContinuousPolicyNetwork(nn.Module):
    """Policy for CONTINUOUS action spaces (e.g., steering angle).
    Outputs a Gaussian distribution (mean and std dev).
    """
    def __init__(self, state_dim, action_dim, hidden=128):
        super().__init__()
        self.base = nn.Sequential(
            nn.Linear(state_dim, hidden),
            nn.ReLU(),
            nn.Linear(hidden, hidden),
            nn.ReLU(),
        )
        self.mean_head = nn.Linear(hidden, action_dim)
        # Log-std is often a separate parameter rather than state-dependent
        self.log_std = nn.Parameter(torch.zeros(1, action_dim))
    
    def forward(self, state):
        features = self.base(state)
        mean = self.mean_head(features)
        std = self.log_std.exp().expand_as(mean)
        return mean, std
    
    def act(self, state):
        state_t = torch.FloatTensor(state).unsqueeze(0)
        mean, std = self.forward(state_t)
        dist = torch.distributions.Normal(mean, std)
        action = dist.sample()
        return action.numpy()[0], dist.log_prob(action).sum(dim=-1)

# Demonstrate Discrete
discrete_policy = DiscretePolicyNetwork(state_dim=4, action_dim=2)
dummy_state = np.random.randn(4)
action_d, log_prob_d = discrete_policy.act(dummy_state)
print(f'Discrete Action Sampled: {action_d} | Log-Prob: {log_prob_d.item():.4f}')

# Demonstrate Continuous
continuous_policy = ContinuousPolicyNetwork(state_dim=4, action_dim=1)
action_c, log_prob_c = continuous_policy.act(dummy_state)
print(f'Continuous Action Sampled: {action_c[0]:.3f} | Log-Prob: {log_prob_c.item():.4f}')


---
## 2 · REINFORCE Algorithm

In [ ]:
class SimpleBalanceEnv:
    """Same environment from RL/03."""
    def __init__(self):
        self.reset()
    def reset(self):
        self.state = np.random.uniform(-0.05, 0.05, size=4)
        self.steps = 0
        return self.state.copy()
    def step(self, action):
        force = 1.0 if action == 1 else -1.0
        self.state[3] += force * 0.02 + np.random.normal(0, 0.01)
        self.state[2] += self.state[3] * 0.02
        self.state[1] += force * 0.01
        self.state[0] += self.state[1] * 0.02
        self.steps += 1
        done = abs(self.state[2]) > 0.5 or abs(self.state[0]) > 2.0 or self.steps >= 200
        reward = 1.0 if not done else 0.0
        return self.state.copy(), reward, done

def reinforce(env, episodes=500, gamma=0.99, lr=1e-3, use_baseline=True):
    """REINFORCE (Monte Carlo Policy Gradient).
    
    Algorithm:
    1. Run a full episode, collecting (state, action, reward)
    2. Compute returns G_t for each step
    3. Compute loss = -Σ log π(a|s) × (G_t - baseline)
    4. Backpropagate and update policy
    
    The baseline reduces variance without changing the gradient's
    expected value (it's still an unbiased estimator).
    """
    policy = DiscretePolicyNetwork(state_dim=4, action_dim=2)
    optimizer = optim.Adam(policy.parameters(), lr=lr)
    
    all_rewards = []
    baseline = 0  # Running average of returns
    
    for episode in range(episodes):
        # --- Collect an episode ---
        state = env.reset()
        log_probs = []
        rewards = []
        
        while True:
            action, log_prob = policy.act(state)
            next_state, reward, done = env.step(action)
            log_probs.append(log_prob)
            rewards.append(reward)
            state = next_state
            if done:
                break
        
        # --- Compute discounted returns ---
        returns = []
        G = 0
        for r in reversed(rewards):
            G = r + gamma * G
            returns.insert(0, G)
        returns = torch.FloatTensor(returns)
        
        # --- Baseline: subtract mean return to reduce variance ---
        if use_baseline:
            baseline = 0.9 * baseline + 0.1 * returns.mean().item()
            advantages = returns - baseline
        else:
            advantages = returns
        
        # Normalize advantages (stabilizes training)
        if len(advantages) > 1:
            advantages = (advantages - advantages.mean()) / (advantages.std() + 1e-8)
        
        # --- Policy gradient update ---
        # Loss = -Σ log π(a|s) × advantage
        # Negative because we MAXIMIZE reward (gradient ASCENT)
        loss = 0
        for log_prob, adv in zip(log_probs, advantages):
            loss += -log_prob * adv
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        all_rewards.append(sum(rewards))
    
    return policy, all_rewards

# Train with and without baseline
env = SimpleBalanceEnv()
print('Training REINFORCE...')
policy_bl, rewards_bl = reinforce(env, episodes=500, use_baseline=True)
print('Training REINFORCE (no baseline)...')
policy_no_bl, rewards_no_bl = reinforce(env, episodes=500, use_baseline=False)

# Compare
fig, ax = plt.subplots(figsize=(10, 5))
window = 30
smoothed_bl = [np.mean(rewards_bl[max(0,i-window):i+1]) for i in range(len(rewards_bl))]
smoothed_no = [np.mean(rewards_no_bl[max(0,i-window):i+1]) for i in range(len(rewards_no_bl))]
ax.plot(smoothed_bl, label='With Baseline', lw=2, color='steelblue')
ax.plot(smoothed_no, label='No Baseline', lw=2, color='coral', alpha=0.7)
ax.set_xlabel('Episode')
ax.set_ylabel('Total Reward (smoothed)')
ax.set_title('REINFORCE: Baseline Reduces Variance')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()
print(f'Baseline version is more stable — less variance in the learning curve.')
print(f'This is why ALL modern PG methods use advantage estimation (GAE in PPO).')

---
## 3 · Actor-Critic: Best of Both Worlds

REINFORCE waits until the episode ends to compute returns (Monte Carlo). Actor-Critic uses a VALUE network to estimate returns at EVERY step (TD learning).

```
ACTOR (Policy Network):  π_θ(a|s) → probabilities → actions
CRITIC (Value Network):  V_φ(s)   → estimated return → tells actor how good its actions are
```

### The Advantage Function

$$A(s_t, a_t) = \underbrace{r_t + \gamma V(s_{t+1})}_{\text{what actually happened}} - \underbrace{V(s_t)}_{\text{what was expected}}$$

- $A > 0$: action was BETTER than expected → increase its probability
- $A < 0$: action was WORSE than expected → decrease its probability
- $A = 0$: exactly as expected → no change

This is the **same advantage** used in PPO (RL/05) and RLHF (RL/06).
### Preventing Deterministic Collapse (Entropy Bonus)

If an action happens to get a lucky positive return early on, the network might immediately shift all probability to that action (e.g., 99% confidence). It stops exploring entirely. This is **deterministic collapse**.

To prevent this, we add an **Entropy Bonus** to the loss function.
Entropy $H(\pi)$ measures how random a distribution is. High entropy = uniform probabilities (exploration). Low entropy = one action dominates 100% (exploitation/collapse).

$$ \text{Loss} = \text{ActorLoss} + \text{CriticLoss} - \beta \cdot H(\pi) $$

By subtracting $H$ from the loss, we explicitly train the network to stay "slightly random" while learning the optimal path.

In [ ]:
class ActorCritic(nn.Module):
    """Actor-Critic Architecture for Discrete actions."""
    def __init__(self, state_dim, action_dim, hidden=128):
        super().__init__()
        # Shared representation
        self.shared = nn.Sequential(
            nn.Linear(state_dim, hidden),
            nn.ReLU()
        )
        # Actor head
        self.actor = nn.Sequential(
            nn.Linear(hidden, action_dim)
        )
        # Critic head
        self.critic = nn.Sequential(
            nn.Linear(hidden, 1)
        )
        
    def forward(self, state):
        features = self.shared(state)
        action_logits = self.actor(features)
        value = self.critic(features)
        return torch.softmax(action_logits, dim=-1), value

def a2c_train_loop(episodes=500, gamma=0.99, lr=2e-3, entropy_beta=0.01):
    env = gym.make('CartPole-v1')
    state_dim = env.observation_space.shape[0]
    action_dim = env.action_space.n
    
    ac = ActorCritic(state_dim, action_dim)
    optimizer = optim.Adam(ac.parameters(), lr=lr)
    
    smoothed_rewards = []
    all_rewards = []
    
    for episode in range(episodes):
        state, _ = env.reset()
        total_reward = 0
        
        # For MC-approximation of advantage (simpler implementation):
        log_probs, values, rewards, entropies = [], [], [], []
        
        while True:
            state_t = torch.FloatTensor(state).unsqueeze(0)
            probs, value = ac(state_t)
            dist = torch.distributions.Categorical(probs)
            
            action = dist.sample()
            log_prob = dist.log_prob(action)
            entropy = dist.entropy()
            
            next_state, reward, terminated, truncated, _ = env.step(action.item())
            done = terminated or truncated
            
            log_probs.append(log_prob)
            values.append(value.squeeze(0))
            rewards.append(reward)
            entropies.append(entropy)
            
            total_reward += reward
            state = next_state
            
            if done:
                break
                
        all_rewards.append(total_reward)
        
        # Calculate Discounted Returns (G_t)
        returns = []
        G = 0
        for r in reversed(rewards):
            G = r + gamma * G
            returns.insert(0, G)
        returns = torch.FloatTensor(returns)
        
        values = torch.cat(values)
        log_probs = torch.cat(log_probs)
        entropies = torch.cat(entropies)
        
        # Advantage = Return - Value Estimate
        advantages = returns - values.detach()
        
        # Total Loss Calculation
        actor_loss = -(log_probs * advantages).mean()
        critic_loss = nn.functional.mse_loss(values, returns)
        entropy_bonus = entropies.mean()
        
        # Beta adjusts exploration/exploitation tradeoff
        loss = actor_loss + critic_loss - (entropy_beta * entropy_bonus)
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
    return all_rewards

print('Training A2C on CartPole-v1... (this will take ~10-20 seconds)')
a2c_rewards = a2c_train_loop(episodes=600, entropy_beta=0.01)

window = 30
smoothed_a2c = [np.mean(a2c_rewards[max(0,i-window):i+1]) for i in range(len(a2c_rewards))]

plt.figure(figsize=(10, 4))
plt.plot(smoothed_a2c, lw=2, color='purple', label='A2C (Smoothed)')
plt.axhline(y=500, color='green', linestyle='--', alpha=0.7, label='Max Reward (500)')
plt.xlabel('Episode')
plt.ylabel('Total Reward')
plt.title('Advantage Actor-Critic (A2C) with Entropy Bonus')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('a2c_cartpole.png')
plt.show()
print(f'Final Average Reward: {np.mean(a2c_rewards[-30:]):.0f}')
print(f'Notice the aggressive steep trajectory once the A2C networks properly align!')

---

## 🔨 Practical Practice

### Exercise 1: Real-Time TD-Advantage
Our A2C uses full Monte-Carlo returns to compute advantages `A = Returns - V(s)`. Rewrite the training loop to compute pure TD-errors at each step: `A = r + γV(s') - V(s)`. Train and check if it learns faster or slower.

### Exercise 2: Tune the Entropy Bonus
Set `entropy_beta = 0.5` and watch it completely fail to converge (too much randomness). Then try `entropy_beta = 0.0001` and track if it gets stuck in sub-optimal local minima. Finding the golden zone is pure ML engineering.

### Exercise 3: Continuous BipedalWalker
Swap `CartPole-v1` for Gymnasium's `BipedalWalker-v3`. Replace `Categorical` dists with `ContinuousPolicyNetwork` using `Normal` distributions. WARNING: This takes tens of thousands of episodes to solve completely via basic PG methods.

---

**Next →** RL 05: Proximal Policy Optimization (PPO)